# AI FAQ Assistant
### PostgreSQL + pgvector + Embeddings + Semantic Search + RAG

---

## Why are we building this?

Normal (keyword) search needs **matching words**. People rarely use the same words as the person who wrote the FAQ.

| The user asks | The best FAQ in our data | Words in common |
|---|---|---|
| *"Where can I store information?"* | *"What is a database?"* | **none** |

A keyword search (`LIKE '%store%'`) will never find this FAQ.
**Semantic search** compares the **meaning** of text instead of the words — and later in this notebook it ranks *"What is a database?"* first for exactly this question.

---

## Project Objective

We will build a simple **AI FAQ Assistant**. The user types a question such as:

> How can computers understand the meaning of text?

Our application will:

```
User Question
      ↓
Create Embedding                 ┐
      ↓                          │
Search PostgreSQL                │  RETRIEVAL
      ↓                          │  (find the right FAQs by meaning)
Find Similar FAQs                │
      ↓                          │
Retrieve Relevant Information    ┘
      ↓
Give Information to Gemini       ┐  GENERATION
      ↓                          │  (write the reply)
Generate Final Answer            ┘
```

This is called **RAG** — **R**etrieval-**A**ugmented **G**eneration: *look the facts up first, then let the model write the answer.*

---

## The tools — four pieces, one notebook

| Tool | Its one job |
|---|---|
| **PostgreSQL** | Stores the 50 FAQs: `id`, `question`, `answer` |
| **pgvector** (extension) | Adds a `VECTOR(768)` column and the `<=>` distance operator |
| **Gemini embeddings** (`gemini-embedding-001`) | Turns any text into 768 numbers |
| **Gemini text model** (`gemini-3.5-flash-lite`) | Writes the final answer from the retrieved FAQs |

**Python** glues it together: `google-genai` talks to Gemini, `psycopg2` talks to PostgreSQL, `pandas` previews the data.

One database stores both the normal data **and** the vectors — no separate vector database is needed.

---

## Workshop roadmap

| Part | What we do |
|---|---|
| **Part 1** | Set up Gemini + PostgreSQL, enable pgvector, load 50 FAQs |
| **Part 2** | Turn every FAQ into a vector (embedding) and store it |
| **Part 3** | Search by meaning with one SQL query |
| **Part 4** | Add Gemini on top → a complete RAG assistant |

---
# Part 1 — Set up PostgreSQL and load the FAQs
---

## Install Libraries

| Library | Used for |
|---|---|
| `google-genai` | Calling Gemini (embeddings + text generation) |
| `psycopg2-binary` | Connecting Python to PostgreSQL |
| `pandas` | Showing our FAQ data as a neat table |

`-q` means *quiet* — it hides the long installation log.

**Before you start:** PostgreSQL must be running, the **pgvector** extension must be installed on the server, and a file `api_key.txt` containing your Gemini API key must sit next to this notebook.

In [7]:
!pip install -q google-genai psycopg2-binary pandas

## Import Libraries

We load the three libraries into this notebook so we can use them in the cells below.

In [8]:
import pandas as pd
import psycopg2
from google import genai

## Gemini API

The **client** is our connection to Gemini. Every Gemini call in this notebook goes through `client`.

 **Keep the API key in `api_key.txt`, never paste it into a notebook you share.**
The code reads the file and `.strip()` removes any extra spaces or newline.
`genai.Client()` creates an object that lets your Python code communicate with Google's Gemini API using your API key.

In [9]:
from google import genai
api_key = open("api_key.txt").read().strip()
client = genai.Client(api_key=api_key)
print("gemini client created sucessfully")

gemini client created sucessfully


## Select Gemini Model

We use **two** Gemini models, each with a different job:

| Variable | Model | Job |
|---|---|---|
| `TEXT_MODEL` | `gemini-3.5-flash-lite` | Writes answers (text generation) |
| `EMBEDDING_MODEL` | `gemini-embedding-001` | Converts text into vectors (embeddings) |

Keep the model names in one variable so you only need to change them in **one place** if the available model changes.

In [11]:
TEXT_MODEL= "gemini-3.5-flash-lite"
EMBEDDING_MODEL= "gemini-embedding-001"

print("Text model" ,TEXT_MODEL)
print(" Embedding model",EMBEDDING_MODEL)

Text model gemini-3.5-flash-lite
 Embedding model gemini-embedding-001


## Test Gemini

A quick check that the API key and model work.
**Expected:** a one-sentence explanation of PostgreSQL.

In [14]:
response = client.models.generate_content(model =TEXT_MODEL,contents = "Explain postgresql in one simple sentence.")
print(response.text)

PostgreSQL is a powerful, open-source relational database management system known for its reliability, advanced features, and ability to handle complex data at scale.


##  Connect to PostgreSQL

These are the details Python needs to find our database:

| Setting | Meaning |
|---|---|
| `DB_HOST` | Where the server runs (`localhost` = this computer) |
| `DB_PORT` | PostgreSQL's default port is **5432** |
| `DB_NAME` | Which database to use |
| `DB_USER` / `DB_PASSWORD` | Login details — change the password to your own |

> In a real project, read the password from an environment variable instead of writing it in the code.

In [17]:
DB_HOST = "localhost" 
DB_PORT = 5432 
DB_NAME = "postgres" 
DB_USER = "postgres" 
DB_PASSWORD = "1234"

##  Create Database Connection



| Object | What it is | What we use it for |
|---|---|---|
| `conn` | The **connection**: an open line between this notebook and the PostgreSQL server | `conn.commit()` saves changes permanently · `conn.rollback()` undoes changes after an error · `conn.close()` ends the connection |
| `cur` | The **cursor**: the tool that sends SQL through that connection and brings results back | `cur.execute(...)` runs a SQL command · `cur.fetchone()` reads one row · `cur.fetchall()` reads all rows |

**Analogy:** `conn` is the phone call to the database, and `cur` is you speaking and listening during the call.

### How a query flows

```
Python  ──cur.execute("SELECT ...")──►  PostgreSQL
Python  ◄──cur.fetchall()────────────   rows come back as a list of tuples
```



**Expected:** `Connected to PostgreSQL successfully!`
If it fails: check the password and that PostgreSQL is running on port 5432.

In [18]:
conn= psycopg2.connect(host = DB_HOST,
                       port = DB_PORT,
                       database= DB_NAME,
                       user = DB_USER,
                       password=DB_PASSWORD)
cur = conn.cursor()
print("connected to postgresql sucessfully!")
                       

connected to postgresql sucessfully!


##  Enable pgvector

```sql
CREATE EXTENSION IF NOT EXISTS vector;
```

This switches pgvector **on inside this database**. After this, PostgreSQL understands:

- a new column type: `VECTOR(n)` — a list of *n* numbers
- new distance operators such as `<=>` (we use it in Part 3)

`IF NOT EXISTS` means running the cell twice does no harm.

In [19]:
cur.execute(""" create extension if not exists vector; """)
conn.commit()
print(" pgvector enabled")

 pgvector enabled


## Create FAQ Table

We will use only **one** table, containing: `id`, `question`, `answer`, `embedding`.

```sql
CREATE TABLE IF NOT EXISTS faqs (
    id         SERIAL PRIMARY KEY,   -- 1, 2, 3 ... assigned automatically
    question   TEXT NOT NULL,
    answer     TEXT NOT NULL,
    embedding  VECTOR(768)           -- exactly 768 numbers
);
```

What the table will look like after we insert data:

| id | question | answer | embedding |
|---|---|---|---|
| 1 | What is PostgreSQL? | PostgreSQL is a free and open-source… | `NULL` (for now) |
| 2 | What is SQL? | SQL stands for Structured Query… | `NULL` (for now) |

- Text and vectors live **side by side in the same row**.
- The `embedding` column starts empty — we fill it in **Part 2**.
- ⚠️ **768** must match the embedding size we ask Gemini for later (`output_dimensionality = 768`), otherwise saving the vectors fails.

In [20]:
cur.execute(""" CREATE TABLE IF NOT EXISTS faqs (
    id         SERIAL PRIMARY KEY,  
    question   TEXT NOT NULL,
    answer     TEXT NOT NULL,
    embedding  VECTOR(768)           
); """)
conn.commit()
print(" FAQ table created !")


 FAQ table created !


## Create 50 FAQs

This is our **knowledge base** — the only information the assistant will answer from.
Each FAQ is a Python dictionary with a `question` and an `answer`.

| Topic | FAQs |
|---|---|
| Databases & SQL | 12 |
| Python & programming | 7 |
| AI, ML & NLP | 5 |
| Embeddings & vectors | 4 |
| Vector databases & pgvector | 3 |
| Semantic search | 4 |
| RAG | 4 |
| Prompts, LLMs & Gemini | 3 |
| APIs & API keys | 2 |
| Docker & cloud | 3 |
| Data in AI applications | 3 |
| **Total** | **50** |

The topics deliberately match this workshop, so we can ask the assistant about what we are learning.

In [21]:
faq_data = [

    {
        "question": "What is PostgreSQL?",
        "answer": "PostgreSQL is a free and open-source relational database system used to store, organize, and manage data."
    },

    {
        "question": "What is SQL?",
        "answer": "SQL stands for Structured Query Language. It is used to work with data stored in databases."
    },

    {
        "question": "What is a database?",
        "answer": "A database is an organized collection of information that can be stored, searched, and managed by a computer system."
    },

    {
        "question": "What is a table in a database?",
        "answer": "A table is a structure inside a database that stores information using rows and columns."
    },

    {
        "question": "What is a row?",
        "answer": "A row represents one complete record in a database table."
    },

    {
        "question": "What is a column?",
        "answer": "A column represents one type or category of information stored in a database table."
    },

    {
        "question": "What is a primary key?",
        "answer": "A primary key is a column that uniquely identifies each row in a database table."
    },

    {
        "question": "What is an index?",
        "answer": "An index is a database structure that helps the database find information more efficiently."
    },

    {
        "question": "What is SELECT in SQL?",
        "answer": "SELECT is an SQL command used to read or retrieve data from a database."
    },

    {
        "question": "What is INSERT in SQL?",
        "answer": "INSERT is an SQL command used to add new records to a database table."
    },

    {
        "question": "What is UPDATE in SQL?",
        "answer": "UPDATE is an SQL command used to change existing data in a database table."
    },

    {
        "question": "What is DELETE in SQL?",
        "answer": "DELETE is an SQL command used to remove records from a database table."
    },

    {
        "question": "What is Python?",
        "answer": "Python is a popular programming language known for its simple syntax and use in software development, data science, and AI."
    },

    {
        "question": "What is a variable in Python?",
        "answer": "A variable is a name used to store a value in a Python program."
    },

    {
        "question": "What is a Python function?",
        "answer": "A function is a reusable block of Python code that performs a particular task."
    },

    {
        "question": "What is a Python list?",
        "answer": "A Python list is a collection that can store multiple values in a single variable."
    },

    {
        "question": "What is a Python dictionary?",
        "answer": "A Python dictionary stores information as key-value pairs."
    },

    {
        "question": "What is programming?",
        "answer": "Programming is the process of writing instructions that tell a computer what to do."
    },

    {
        "question": "What is a programming language?",
        "answer": "A programming language is a language used by programmers to write instructions that computers can execute."
    },

    {
        "question": "What is an API?",
        "answer": "An API is a way for one software application to communicate with another software application."
    },

    {
        "question": "What is an API key?",
        "answer": "An API key is a piece of information used by an application to identify or authenticate access to an API."
    },

    {
        "question": "What is Artificial Intelligence?",
        "answer": "Artificial Intelligence, or AI, is technology that allows computers to perform tasks that normally require human-like intelligence."
    },

    {
        "question": "What is Machine Learning?",
        "answer": "Machine Learning is a part of AI where computers learn patterns from data and use those patterns to make predictions or decisions."
    },

    {
        "question": "What is an AI model?",
        "answer": "An AI model is a computer system that has learned patterns from data and can perform tasks such as generating text or making predictions."
    },

    {
        "question": "What is Natural Language Processing?",
        "answer": "Natural Language Processing, or NLP, is a field of AI that helps computers work with human language."
    },

    {
        "question": "What is text generation?",
        "answer": "Text generation is the process of using an AI model to create human-readable text from an input."
    },

    {
        "question": "What is an embedding?",
        "answer": "An embedding is a list of numbers that represents information such as text in a form that an AI system can process."
    },

    {
        "question": "What is a vector?",
        "answer": "A vector is a list of numbers. In AI applications, vectors can be used to represent information such as the meaning of text."
    },

    {
        "question": "Why do we use embeddings?",
        "answer": "Embeddings allow computers to represent the meaning of information as numbers so that similar information can be compared."
    },

    {
        "question": "How does text become an embedding?",
        "answer": "An embedding model takes text as input and converts it into a list of numbers that represents information about the text."
    },

    {
        "question": "What is a vector database?",
        "answer": "A vector database is a database system designed to store and search vector representations of information."
    },

    {
        "question": "What is pgvector?",
        "answer": "pgvector is a PostgreSQL extension that allows PostgreSQL to store and search vector data."
    },

    {
        "question": "Why use pgvector with PostgreSQL?",
        "answer": "pgvector allows PostgreSQL to store normal data together with vector data and perform similarity searches."
    },

    {
        "question": "What is semantic search?",
        "answer": "Semantic search finds information based on meaning instead of only looking for exact matching words."
    },

    {
        "question": "How is semantic search different from keyword search?",
        "answer": "Keyword search mainly looks for matching words, while semantic search uses embeddings to compare the meaning of information."
    },

    {
        "question": "What is similarity search?",
        "answer": "Similarity search finds information whose vector representation is similar to the vector representation of the user's question."
    },

    {
        "question": "Why are vectors useful for search?",
        "answer": "Vectors allow information to be represented as numbers so that a system can compare the similarity between different pieces of information."
    },

    {
        "question": "What is RAG?",
        "answer": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information and gives that information to an AI model to generate an answer."
    },

    {
        "question": "How does RAG work?",
        "answer": "RAG first searches a knowledge source for relevant information and then gives the retrieved information to an AI model before generating the final answer."
    },

    {
        "question": "Why do we use RAG?",
        "answer": "RAG allows an AI application to use information retrieved from a knowledge source when generating an answer."
    },

    {
        "question": "What is a RAG application?",
        "answer": "A RAG application is an AI application that retrieves relevant information from a knowledge source and uses that information to generate an answer."
    },

    {
        "question": "What is a prompt?",
        "answer": "A prompt is an instruction or question given to an AI model."
    },

    {
        "question": "What is a language model?",
        "answer": "A language model is an AI model that can process and generate human language."
    },

    {
        "question": "What is Gemini?",
        "answer": "Gemini is a family of AI models from Google that can be used for tasks such as understanding and generating content."
    },

    {
        "question": "What is Docker?",
        "answer": "Docker is a technology used to package applications and their dependencies into containers."
    },

    {
        "question": "What is a Docker container?",
        "answer": "A Docker container is an isolated environment that contains an application and the components it needs to run."
    },

    {
        "question": "What is cloud computing?",
        "answer": "Cloud computing means using computing resources such as servers and storage over the internet."
    },

    {
        "question": "What is data?",
        "answer": "Data is information that can be stored, processed, and analyzed by a computer."
    },

    {
        "question": "Why do AI applications need databases?",
        "answer": "AI applications can use databases to store information, retrieve useful data, and provide information to AI models."
    },

    {
        "question": "How can PostgreSQL be used in an AI application?",
        "answer": "PostgreSQL can store application data and, with pgvector, can also store and search vector embeddings for AI applications."
    }

]

## Check Number of FAQs

**Expected:** `Number of FAQs: 50`

In [23]:
print(" number of faqs:", len(faq_data))

 number of faqs: 50


##  Display Data

`pandas` shows the list of dictionaries as a table — an easy way to eyeball the data before it goes into the database.

In [24]:
df = pd.DataFrame(faq_data)
df

,question,answer
0,What is PostgreSQL?,PostgreSQL is a free and open-source relationa...
1,What is SQL?,SQL stands for Structured Query Language. It i...
2,What is a database?,A database is an organized collection of infor...
3,What is a table in a database?,A table is a structure inside a database that ...
4,What is a row?,A row represents one complete record in a data...
5,What is a column?,A column represents one type or category of in...
6,What is a primary key?,A primary key is a column that uniquely identi...
7,What is an index?,An index is a database structure that helps th...
8,What is SELECT in SQL?,SELECT is an SQL command used to read or retri...
9,What is INSERT in SQL?,INSERT is an SQL command used to add new recor...


##  Insert 50 FAQs into PostgreSQL

One `INSERT` per FAQ. The `%s` placeholders let psycopg2 insert the values **safely** (no manual quoting, no SQL injection).
We only insert `question` and `answer` — `id` is created automatically and `embedding` stays empty for now.
`conn.commit()` saves everything permanently.

> ⚠️ **Run this cell only once.** Running it again adds 50 duplicate rows (ids 51–100) that will never get embeddings.
> To start over, first run: `cur.execute("TRUNCATE faqs RESTART IDENTITY;"); conn.commit()`

In [25]:
for item in faq_data:
    cur.execute(""" insert into faqs (question, answer) values (%s, %s)""", (item["question"], item["answer"]))
conn.commit()

## Check Database

`SELECT COUNT(*)` counts the rows in the table.
**Expected:** `Number of FAQs in PostgreSQL: 50`

In [26]:
cur.execute(""" select count(*) from faqs;""")
count = cur.fetchone()[0]
print(" Number of faqs in postgresql:",count)

 Number of faqs in postgresql: 50


##  View FAQ Data

Read every row back from PostgreSQL to confirm the data is really stored there.

In [29]:
cur.execute(""" select id,question,answer from faqs ;""")
rows = cur.fetchall()
#print(rows)
for row in rows:
    print("ID:",row[0])
    print("Question:",row[1])
    print("Answer:",row[2])
    print("-" * 70)
    

ID: 1
Question: What is PostgreSQL?
Answer: PostgreSQL is a free and open-source relational database system used to store, organize, and manage data.
----------------------------------------------------------------------
ID: 2
Question: What is SQL?
Answer: SQL stands for Structured Query Language. It is used to work with data stored in databases.
----------------------------------------------------------------------
ID: 3
Question: What is a database?
Answer: A database is an organized collection of information that can be stored, searched, and managed by a computer system.
----------------------------------------------------------------------
ID: 4
Question: What is a table in a database?
Answer: A table is a structure inside a database that stores information using rows and columns.
----------------------------------------------------------------------
ID: 5
Question: What is a row?
Answer: A row represents one complete record in a database table.
-----------------------------------

---
# Part 2 — Turn every FAQ into a vector
---

## What is an embedding?

An **embedding** is *meaning written as numbers*.

```
"What is PostgreSQL?"  ──►  embedding model  ──►  [-0.0078, 0.0034, -0.0021, -0.0651, -0.0217, … ]
                                                    └──────────── 768 numbers ────────────┘
```

| Idea | Explanation |
|---|---|
| **Fixed length** | Every text — short or long — becomes exactly **768** numbers |
| **Meaning → direction** | Texts with similar meaning get vectors that point the same way |
| **Learned, not written** | Gemini's model learned these numbers from huge amounts of text |

No single number means anything on its own — the **pattern** as a whole encodes the meaning.

### Similar meanings sit close together

A sketch only (real embeddings have 768 dimensions, squeezed here into 2):

```
        ★ "Where can I store information?"   ← your question
         ╲
          ● What is a database?
            ● What is a table in a database?
          ● What is SQL?                              ● What is a vector?
                                                          ● Why do we use embeddings?
     ● What is Python?                               ● How does text become an embedding?
```

A new question is embedded the same way and lands somewhere in this space.
Its **nearest neighbours** are the most relevant FAQs. That is all semantic search is.

---

## Create Embedding Function

We will use Gemini's embedding model.

- `contents=text` → the text to convert
- `output_dimensionality: 768` → must match our `VECTOR(768)` column
- `result.embeddings[0].values` → a plain Python list of 768 decimal numbers

In [32]:
def create_embedding(text):
    result = client.models.embed_content(
        model = EMBEDDING_MODEL,
        contents =text,
        config={
            "output_dimensionality":768
        }
    )
    return result.embeddings[0].values

##  Test Embedding

**Expected:** `Number of values: 768`, followed by the first 10 numbers of the vector.

In [36]:
text =" "
embedding = create_embedding(text)
print(" Number of values:", len(embedding))
print(" First 10 values:", embedding[:10])

 Number of values: 768
 First 10 values: [-0.0053882604, 0.0076578804, 0.011579545, -0.0575647, 0.029505298, 0.011536011, -0.0020401385, 0.003505577, -0.007628863, 0.0032300877]


##  Create Text for FAQ Embedding

We will combine the **question and answer** into one piece of text before embedding it.

Why both? The vector then carries the **full meaning** of the FAQ, not just the question — so a user question can match on words or ideas that appear only in the answer.

In [37]:
def faq_text(item):
    return f"""
    question:
    {item["question"]}
    Answer:
    {item["answer"]}
    """
    

## Test FAQ Text

See exactly what text will be sent to the embedding model for FAQ #1.

In [39]:
sample = faq_text(faq_data[1])
print(sample)


    question:
    What is SQL?
    Answer:
    SQL stands for Structured Query Language. It is used to work with data stored in databases.
    


## Create Embeddings for All 50 FAQs

For each FAQ the loop does:

```
Python list            [0.012, -0.013, …]
      ↓   convert to text
Text for SQL           "[0.012,-0.013,…]"      ← the format pgvector understands
      ↓   UPDATE faqs SET embedding = … WHERE id = …
PostgreSQL column      VECTOR(768)
```

- `enumerate` gives position `i` = 0, 1, 2 …, so `faq_id = i + 1` matches `id` 1, 2, 3 …
  (this works because the table was filled once, in the same order).
- **One API call per FAQ** — this cell takes a little while. Watch the `Processing FAQ n of 50` lines.

In [41]:
for i,item in enumerate(faq_data):
    print(f"Processing FAQ {i+1} of {len(faq_data)}")
    text = faq_text(item)
    embedding= create_embedding(text)
    faq_id = i + 1
    embedding_string = "[" + ",".join( str(x) for x in embedding ) + "]" 
    cur.execute(""" update faqs set embedding = %s where id = %s""",
                (embedding_string,faq_id))
    conn.commit()
print("All faqs embedding stored sucesfully")
    

Processing FAQ 1 of 50
Processing FAQ 2 of 50
Processing FAQ 3 of 50
Processing FAQ 4 of 50
Processing FAQ 5 of 50
Processing FAQ 6 of 50
Processing FAQ 7 of 50
Processing FAQ 8 of 50
Processing FAQ 9 of 50
Processing FAQ 10 of 50
Processing FAQ 11 of 50
Processing FAQ 12 of 50
Processing FAQ 13 of 50
Processing FAQ 14 of 50
Processing FAQ 15 of 50
Processing FAQ 16 of 50
Processing FAQ 17 of 50
Processing FAQ 18 of 50
Processing FAQ 19 of 50
Processing FAQ 20 of 50
Processing FAQ 21 of 50
Processing FAQ 22 of 50
Processing FAQ 23 of 50
Processing FAQ 24 of 50
Processing FAQ 25 of 50
Processing FAQ 26 of 50
Processing FAQ 27 of 50
Processing FAQ 28 of 50
Processing FAQ 29 of 50
Processing FAQ 30 of 50
Processing FAQ 31 of 50
Processing FAQ 32 of 50
Processing FAQ 33 of 50
Processing FAQ 34 of 50
Processing FAQ 35 of 50
Processing FAQ 36 of 50
Processing FAQ 37 of 50
Processing FAQ 38 of 50
Processing FAQ 39 of 50
Processing FAQ 40 of 50
Processing FAQ 41 of 50
Processing FAQ 42 of 50
P

## Check Embeddings

Count rows whose `embedding` is filled in.
**Expected:** `FAQs with embeddings: 50`

In [42]:
cur.execute("""
SELECT
    COUNT(*)
FROM faqs
WHERE embedding IS NOT NULL;
""")

embedding_count = cur.fetchone()[0]

print(
    "FAQs with embeddings:",
    embedding_count
    
)

FAQs with embeddings: 50


## Check One Embedding

Read FAQ #1 back — the vector is really stored **inside PostgreSQL**, next to the question and answer.

In [49]:
cur.execute(""" select id,question,embedding from faqs where id =1;
""")
row = cur.fetchone()
print(row)

(1, 'What is PostgreSQL?', '[0.0011159262,-0.009541954,-0.004197898,-0.052550964,-0.028240293,0.0237385,0.009727532,0.035951633,0.030354735,-0.001988467,-0.0023682853,0.008602182,-0.0091249915,0.009066273,0.11384246,0.0005421942,0.003477904,0.0008572842,0.0043076538,-0.029275699,-0.028536214,-0.010606136,0.0046418533,-0.02266282,-0.015665727,-0.03139092,0.028778795,-0.029700303,0.01868088,0.037414182,-0.005404819,0.0014363902,-0.0012849909,0.022855578,0.0087461835,0.014714352,0.002389843,0.004127637,-0.02179305,0.008740683,-0.010609924,0.006199944,-0.012129536,-0.007333983,5.3062042e-05,0.0022701183,-0.00026485464,-0.012870145,-0.02371679,0.030058697,-0.010557497,-0.03765141,-0.020572172,-0.17100106,0.013420084,0.011040458,0.0022345264,0.0012935397,0.005080204,-0.020832608,-0.016359026,-0.02373943,0.00878185,-0.010572272,-0.011543744,0.00881024,0.025009181,-0.024578659,-0.022989197,0.009475658,-0.002628571,0.011052666,0.0052388036,-0.0015561724,-0.019964598,-0.024998847,0.034099042,-0.

In [48]:
conn.rollback()

---
# Part 3 — Search by meaning with pgvector
---

## How does pgvector compare two meanings?

pgvector measures the **distance** between two vectors.

| Operator | Measures | In this notebook |
|---|---|---|
| `<=>` | **Cosine distance** | ✅ **Used — our choice** |
| `<->` | Euclidean (L2) distance | not used |
| `<#>` | Negative inner product | not used |

- **Cosine distance = 1 − cosine similarity.** It compares *where* two vectors point, not how long they are.
- **0 = same direction.** **Smaller distance = closer meaning.**
- Sort by distance, smallest first → the best FAQ comes out on top.

---

## Create Semantic Search Function

Semantic search is **one SQL query**:

| Step | What happens |
|---|---|
| 1. Embed the question | Using the **same** model as the FAQs — otherwise the vectors can't be compared |
| 2. `ORDER BY embedding <=> question_vector` | Nearest meaning first |
| 3. `LIMIT top_k` | Keep the best 3 |

`%s::vector` converts our text `"[...]"` into a pgvector value.
The vector is passed **twice**: once to *show* the distance, once to *sort* by it.

In [50]:
def search_faq(question, top_k=3):
    # Create embedding for the user's question
    query_embedding= create_embedding(question)
    # convert python list  into postgresql vector format
    embedding_string = "[" + ",".join(
        str(x) for x in query_embedding
    ) + "]"

    #search in the postgresql
    cur.execute(""" select question,answer,
                embedding <=>%s::vector as distance
                from faqs
                where embedding is not NULL
                order by embedding <=>%s::vector
                limit %s; """
                ,(embedding_string,embedding_string,top_k)
               )
    return cur.fetchall()

##  Test Semantic Search

An easy case first: the question exactly matches an FAQ.
**Expected:** *"What is a database?"* first, with the smallest distance.

In [52]:
results = search_faq( "What is a database?", top_k=3 )
for result in results: 
    print("Question:") 
    print(result[0]) 
    print("\nAnswer:") 
    print(result[1]) 
    print("\nDistance:")
    print(result[2]) 
    print("=" * 70)

Question:
What is a database?

Answer:
A database is an organized collection of information that can be stored, searched, and managed by a computer system.

Distance:
0.23381876691584313
Question:
What is a table in a database?

Answer:
A table is a structure inside a database that stores information using rows and columns.

Distance:
0.307933105713479
Question:
What is SQL?

Answer:
SQL stands for Structured Query Language. It is used to work with data stored in databases.

Distance:
0.33446260558247976


##  Test Semantic Search — no shared words

This is the example from the start of the notebook. *"Where can I store information?"* shares **no words** with *"What is a database?"*.

**Expected result (from our run):**

| # | Retrieved FAQ | Distance |
|---|---|---|
| 1 | What is a database? | 0.345 |
| 2 | What is data? | 0.351 |
| 3 | What is a table in a database? | 0.373 |

The match is on **meaning**, not words.

In [54]:
results = search_faq( "What can i stored", top_k=3 )
for result in results: 
    print("Question:") 
    print(result[0]) 
    print("\nAnswer:") 
    print(result[1]) 
    print("\nDistance:")
    print(result[2]) 
    print("=" * 70)

Question:
What is data?

Answer:
Data is information that can be stored, processed, and analyzed by a computer.

Distance:
0.4221074603892564
Question:
What is a Python dictionary?

Answer:
A Python dictionary stores information as key-value pairs.

Distance:
0.4536225956673564
Question:
What is SQL?

Answer:
SQL stands for Structured Query Language. It is used to work with data stored in databases.

Distance:
0.45542807203554214


In [57]:
results = search_faq( "today is suny?", top_k=3 )
for result in results: 
    print("Question:") 
    print(result[0]) 
    print("\nAnswer:") 
    print(result[1]) 
    print("\nDistance:")
    print(result[2]) 
    print("=" * 70)

Question:
What is an API key?

Answer:
An API key is a piece of information used by an application to identify or authenticate access to an API.

Distance:
0.5246583630903328
Question:
What is RAG?

Answer:
RAG stands for Retrieval-Augmented Generation. It retrieves relevant information and gives that information to an AI model to generate an answer.

Distance:
0.5286916219421128
Question:
What is Gemini?

Answer:
Gemini is a family of AI models from Google that can be used for tasks such as understanding and generating content.

Distance:
0.5332790502161064


## Test Embedding Search

The model understands that *"represent the meaning of text"* is about vectors and embeddings:

| # | Retrieved FAQ | Distance |
|---|---|---|
| 1 | What is a vector? | 0.268 |
| 2 | Why do we use embeddings? | 0.292 |
| 3 | How does text become an embedding? | 0.298 |

👉 Try your own question: `search_faq("your question here")`

---
# Part 4 — Add Gemini: Retrieval-Augmented Generation (RAG)
---

Search gives us the right FAQs. Now Gemini turns them into a natural answer.

| Letter | Step | Function in this notebook |
|---|---|---|
| **R** | **Retrieve** | `search_faq()` finds the 3 FAQs closest in meaning |
| **A** | **Augment** | `get_context()` pastes them into the prompt as context |
| **G** | **Generate** | `generate_answer()` — Gemini writes a short answer using that context |

**Why RAG?** The model answers from **our** data instead of guessing, and we can show exactly which FAQs it used.
Gemini never searches the database itself — **we** give it the facts.

---

## Test RAG Retrieval

Create a function that collects the relevant information.
`get_context()` runs the search and joins the top 3 question/answer pairs into one block of text.

In [58]:
def get_context(question):
    results = search_faq(question,top_k=3)
    context=""
    for result in results:
        context +=f"""
        Question:{result[0]}
        Answer: {result[1]}"""
    return context

## See the RAG Context

This is exactly the text Gemini will receive as "Retrieved Information".
For *"What is pgvector?"* we expect the three pgvector-related FAQs.

In [60]:
question = " What is pgvector"
context =get_context(question)
print(context)


        Question:What is pgvector?
        Answer: pgvector is a PostgreSQL extension that allows PostgreSQL to store and search vector data.
        Question:Why use pgvector with PostgreSQL?
        Answer: pgvector allows PostgreSQL to store normal data together with vector data and perform similarity searches.
        Question:How can PostgreSQL be used in an AI application?
        Answer: PostgreSQL can store application data and, with pgvector, can also store and search vector embeddings for AI applications.


## Generate Final Gemini Answer

The prompt keeps Gemini **grounded** in our data:

| Part of the prompt | Filled with |
|---|---|
| `{context}` | The top 3 FAQs from `get_context()` |
| `{question}` | Exactly what the user typed |
| Instructions 1, 2, 4, 5 | Control tone and length (simple, short, beginner-friendly) |
| **Instruction 3: "Do not invent information."** | **The guardrail** — stay inside the context |

In [61]:
def generate_answer(question,context):
    prompt= f"""
    you are helpful beginner-frindly AI assistant
    use the information below to answer the user question.
    retrieved information:
    {context}
    user Question:
    {question}

    Instruction:
    1. Answer in simple english
    2. use the retrieved information
    3. Do not invent information
    4. keep the answer suitable for a beginner
    5. Give a clear and short explanation
    """
    response = client.models.generate_content(
        model = TEXT_MODEL,
        contents=prompt
    )
    return response.text

##  Test Gemini With RAG

Retrieve the context, then let Gemini answer from it.
Compare the answer with the context printed above — it should use the same facts.

In [63]:
question = " what is database "
context = get_context(question)
answer = generate_answer(question,context)
print("Question")
print(question)
print(answer)

Question
 what is database 
A database is an organized collection of information that can be stored, searched, and managed by a computer system.


# Create Complete AI Function

Now we combine Search + RAG + Gemini into one function:

```
ask_ai(question)  =  get_context(question)  +  generate_answer(question, context)
                     └──── retrieval ────┘     └──────── generation ────────┘
```

Retrieval and generation — in two lines.

In [68]:
def ask_ai(question):
    # step 1 retriev relevant information
    context = get_context(question)
    #step 2 generate answer using gemini
    answer= generate_answer(question,context)
    return answer

## Test Complete AI Assistant

One call, one answer.

In [69]:
question = "What is PostgreSQL?"

answer = ask_ai(question)

print("AI:")
print(answer)

AI:
PostgreSQL is a free and open-source system used to store, organize, and manage your data.


##  Test 10 Questions

Run the assistant over a list of questions to see it answer consistently.

In [70]:
questions = [

    "What is PostgreSQL?",

    "What is SQL?",

    "What is a database?",

    "What is Python?",

    "What is Artificial Intelligence?",

    "What is Machine Learning?",

    "What is an embedding?",

    "What is pgvector?",

    "What is semantic search?",

    "What is RAG?"

]

for question in questions:

    print("=" * 70)

    print("QUESTION:")
    print(question)

    print("\nAI ANSWER:")

    print(ask_ai(question))

    print()

QUESTION:
What is PostgreSQL?

AI ANSWER:
PostgreSQL is a free and open-source system used to store, organize, and manage data.

QUESTION:
What is SQL?

AI ANSWER:
SQL stands for Structured Query Language. It is used to work with data stored in databases.

QUESTION:
What is a database?

AI ANSWER:
A database is an organized collection of information that can be stored, searched, and managed by a computer system.

QUESTION:
What is Python?

AI ANSWER:
Python is a popular programming language that is easy to read and write. It is commonly used for software development, data science, and AI.

QUESTION:
What is Artificial Intelligence?

AI ANSWER:
Artificial Intelligence, or AI, is technology that allows computers to do tasks that usually need human-like intelligence.

QUESTION:
What is Machine Learning?

AI ANSWER:
Machine Learning is a part of AI where computers learn patterns from data and use those patterns to make predictions or decisions.

QUESTION:
What is an embedding?

AI ANSWER:


##  Test Natural Questions

The user does not have to use exactly the same words as the FAQ.
In the next cells none of the questions matches an FAQ word-for-word — semantic search still finds the right information.

In [71]:
question = "How can I store information?"

print("Question:")
print(question)

print("\nAI:")
print(ask_ai(question))

Question:
How can I store information?

AI:
Based on the information provided, you can store information using:

* **A computer:** Computers can store, process, and analyze information (data).
* **A database:** This is an organized collection of information that a computer system can store, search, and manage.
* **A table:** This is a structure inside a database that stores information using rows and columns.


In [72]:
question = "How can computer understant text"

print("Question:")
print(question)

print("\nAI:")
print(ask_ai(question))

Question:
How can computer understant text

AI:
Computers understand text through a field of AI called Natural Language Processing (NLP), which helps computers work with human language. 

To do this, an embedding model takes the text as input and converts it into a list of numbers that represents information about the text.


##  Show Retrieved Information

RAG is **explainable**: we can always show which FAQs were used to build an answer.
This function prints the top 3 retrieved FAQs and their distances.

In [74]:
def show_search_results(question):

    results = search_faq(
        question,
        top_k=3
    )

    print("USER QUESTION")
    print(question)

    print("\nRETRIEVED INFORMATION")
    print("=" * 70)

    for i, result in enumerate(results):

        print(f"\nRESULT {i + 1}")

        print("\nQuestion:")
        print(result[0])

        print("\nAnswer:")
        print(result[1])

        print("\nDistance:")
        print(result[2])

        print("-" * 70)

##  Test Search Results

For *"How can computers understand text?"* we expect: NLP, how text becomes an embedding, and text generation.

In [76]:

show_search_results(
    "can you explain computer understanding text"
)

USER QUESTION
can you explain computer understanding text

RETRIEVED INFORMATION

RESULT 1

Question:
How does text become an embedding?

Answer:
An embedding model takes text as input and converts it into a list of numbers that represents information about the text.

Distance:
0.29744652695499574
----------------------------------------------------------------------

RESULT 2

Question:
What is Natural Language Processing?

Answer:
Natural Language Processing, or NLP, is a field of AI that helps computers work with human language.

Distance:
0.30679235009935857
----------------------------------------------------------------------

RESULT 3

Question:
What is text generation?

Answer:
Text generation is the process of using an AI model to create human-readable text from an input.

Distance:
0.32594168247645594
----------------------------------------------------------------------


##  Show Complete RAG Process

One question, end to end — this function prints every step:

| Step | What happens |
|---|---|
| 1 · Question | The user's text |
| 2 · Embed | 768 numbers for the question |
| 3 · Search | pgvector finds the 3 nearest FAQs |
| 4 · Context | Those 3 Q&A pairs are pasted into the prompt |
| 5 · Gemini | Reads the context and the question |
| 6 · Answer | The final reply |

In [78]:
def explain_rag_process(question):

    print("=" * 70)
    print("STEP 1 — USER QUESTION")
    print("=" * 70)

    print(question)

    print("\n")

    print("=" * 70)
    print("STEP 2 — CREATE QUESTION EMBEDDING")
    print("=" * 70)

    query_embedding = create_embedding(question)

    print(
        "Embedding created."
    )

    print(
        "Number of values:",
        len(query_embedding)
    )

    print("\n")

    print("=" * 70)
    print("STEP 3 — SEARCH POSTGRESQL")
    print("=" * 70)

    results = search_faq(
        question,
        top_k=3
    )

    for i, result in enumerate(results):

        print(f"\nRESULT {i + 1}")

        print("Question:")
        print(result[0])

        print("\nAnswer:")
        print(result[1])

        print("\nDistance:")
        print(result[2])

    print("\n")

    print("=" * 70)
    print("STEP 4 — CREATE RAG CONTEXT")
    print("=" * 70)

    context = get_context(question)

    print(context)

    print("=" * 70)
    print("STEP 5 — SEND CONTEXT TO GEMINI")
    print("=" * 70)

    answer = generate_answer(
        question,
        context
    )

    print("=" * 70)
    print("STEP 6 — FINAL AI ANSWER")
    print("=" * 70)

    print(answer)

##  Run Complete RAG Demo

Watch the six steps print one after another.

In [79]:
explain_rag_process(
    "How can computers understand the meaning of text?"
)

STEP 1 — USER QUESTION
How can computers understand the meaning of text?


STEP 2 — CREATE QUESTION EMBEDDING
Embedding created.
Number of values: 768


STEP 3 — SEARCH POSTGRESQL

RESULT 1
Question:
What is Natural Language Processing?

Answer:
Natural Language Processing, or NLP, is a field of AI that helps computers work with human language.

Distance:
0.2989670121579673

RESULT 2
Question:
What is semantic search?

Answer:
Semantic search finds information based on meaning instead of only looking for exact matching words.

Distance:
0.31577259252741974

RESULT 3
Question:
How does text become an embedding?

Answer:
An embedding model takes text as input and converts it into a list of numbers that represents information about the text.

Distance:
0.3206323465319826


STEP 4 — CREATE RAG CONTEXT

        Question:What is Natural Language Processing?
        Answer: Natural Language Processing, or NLP, is a field of AI that helps computers work with human language.
        Question:Wh

---
# Live demo — your turn!
---

##  Interactive FAQ Assistant

Run the cell and type questions. Type **`exit`** to stop.

Try:
- *What is pgvector?*
- *How can I search by meaning?*
- *Why do AI applications need databases?*
- **An off-topic question** — the retrieved FAQs won't help. Does Gemini admit it, or drift?

In [77]:
while True:
    question = input("\nAsk a quesiton:")
    if question.lower() == "exit":
        print("goodBye!")
        break
    answer = ask_ai(question)
    print("\nAI:")
    print(answer)


Ask a quesiton: PostgreSQL is a free and open-source system used to store, organize, and manage data. Is this true?



AI:
Yes, that is true! 

According to the information provided, PostgreSQL is a free and open-source relational database system used to store, organize, and manage data.



Ask a quesiton: exit


goodBye!


## Complete Project Test

A final check over 20 questions — some exact, some reworded.

---
# Wrap-up
---

## Recap

| Piece | Role |
|---|---|
| **PostgreSQL** | Stores the knowledge |
| **pgvector** | Stores vectors and finds the nearest ones by meaning (`<=>`) |
| **Gemini** | Creates embeddings and writes the final answer |
| **RAG** | Ties them together: retrieve → augment → generate |

## Taking it beyond 50 FAQs

| Idea | How |
|---|---|
| **Add an index** | `CREATE INDEX ON faqs USING hnsw (embedding vector_cosine_ops);` — keeps search fast with thousands or millions of rows |
| **Batch the API calls** | Embed many texts per request instead of one at a time |
| **Use real documents** | Split PDFs, lecture notes or forum posts into chunks — one row per chunk |
| **Keep secrets out** | Read the DB password and API key from environment variables |

## Questions?

In [ ]:
conn.close()